# Supplementary Figure S1 - dataset overlap, split by serotype

Produces `figS1.pdf`, the only panel from the original upset notebook that the paper uses.

The three source tables ship with the repository in `data/figure_inputs.tar.zst`; `rawpath`
extracts them on first use, so no prediction snapshot is needed.


In [ ]:
import sys
sys.path.insert(0, '..')
import rawpath as rp
import pandas as pd
from functools import reduce
from upsetplot import UpSet, from_memberships
from matplotlib import pyplot as plt
from matplotlib import cm

# 1) source tables
df1 = pd.read_csv(rp.at('250511/2_dataset/1_full_bal/hum_ani_full.csv'))
df2 = pd.read_csv(rp.at('250520/1_dataset/3_ms_ql/hum_ani_full.csv'))
df2 = df2[df2['Target'] == 1]
df3 = pd.read_csv(rp.at('250511/2_dataset/2_ic50/hum_ani_full.csv'))
df3['Target'] = df3['Target_500']

# 2) serotype label
def add_hla_type(df):
    human = df[df['HLA_Name'].str.startswith('HLA-')].copy()
    mouse = df[df['HLA_Name'].str.startswith('H2-')].copy()
    human['Type'] = human['HLA_Name'].str[:6]  # ex: 'HLA-DR'
    mouse['Type'] = 'H2'
    return pd.concat([human, mouse], axis=0)

dfs = [add_hla_type(df) for df in (df1, df2, df3)]
dataset_names = ['Qualitative', 'MS', 'IC50']

# 3) add a membership boolean column to each frame
def annotate(df, name):
    tmp = df[['HLA_Name','Epi_Seq','Type']].copy()
    tmp[name] = True
    return tmp

annotated = [annotate(df, nm) for df, nm in zip(dfs, dataset_names)]

df_all = reduce(
    lambda left, right: pd.merge(left, right, on=['HLA_Name','Epi_Seq','Type'], how='outer'),
    annotated
)

# 5) NaN → False
for nm in dataset_names:
    df_all[nm] = df_all[nm].fillna(False)

df_up = df_all.set_index(dataset_names)

In [ ]:
from svgutils.compose import Figure, SVG
import xml.etree.ElementTree as ET

# Read the declared width and height out of an SVG
def get_svg_size(svg_file):
    tree = ET.parse(svg_file)
    root = tree.getroot()
    width = float(root.attrib['width'].replace('pt','').replace('px','').replace('cm',''))
    height = float(root.attrib['height'].replace('pt','').replace('px','').replace('cm',''))
    return width, height
import cairosvg

In [ ]:
from matplotlib import cm

def get_memberships(dfs, dataset_names, target_val):
    memberships = {}
    for df, name in zip(dfs, dataset_names):
        filtered = df[df['HLA_Name'].str.startswith(target_val)]
        keys = set(zip(filtered['HLA_Name'], filtered['Epi_Seq']))
        memberships[name] = keys
    return memberships


dp_memberships = get_memberships(dfs, dataset_names, 'HLA-DP')
dq_memberships = get_memberships(dfs, dataset_names, 'HLA-DQ')
dr_memberships = get_memberships(dfs, dataset_names, 'HLA-DR')
h2_memberships = get_memberships(dfs, dataset_names, 'H2')

def create_upset_data(memberships):
    all_keys = set.union(*memberships.values())
    data = []
    for key in all_keys:
        present = [name for name, keys in memberships.items() if key in keys]
        data.append(present)
    upset_data = from_memberships(data)
    upset_data = upset_data.groupby(upset_data.index.names).size()
    return upset_data

dp_upset_data = create_upset_data(dp_memberships)
dq_upset_data = create_upset_data(dq_memberships)
dr_upset_data = create_upset_data(dr_memberships)
h2_upset_data = create_upset_data(h2_memberships)

# Positive UpSet Plot
colors = cm.Accent.colors
def to_hex(color):
    return '#{:02x}{:02x}{:02x}'.format(int(color[0]*255), int(color[1]*255), int(color[2]*255))
colors = [to_hex(color) for color in cm.Set2.colors]
UpSet(dp_upset_data, show_counts=True, facecolor=colors[1], element_size=30).plot()
# label = 'c'
# plt.text(-2.5, 1.10, label, transform=plt.gca().transAxes, fontweight='bold', va='bottom', ha='right', fontsize=12)
plt.suptitle('HLA-DP')
plt.savefig('4c1.svg', bbox_inches='tight', dpi=600)
# plt.show()

UpSet(dq_upset_data, show_counts=True, facecolor=colors[2], element_size=30).plot()
plt.suptitle('HLA-DQ')
plt.savefig('4c2.svg', bbox_inches='tight', dpi=600)
# plt.show()

UpSet(dr_upset_data, show_counts=True, facecolor=colors[3], element_size=30).plot()
plt.suptitle('HLA-DR')
plt.savefig('4c3.svg', bbox_inches='tight', dpi=600)
# plt.show()

UpSet(h2_upset_data, show_counts=True, facecolor=colors[0], element_size=30).plot()
plt.suptitle('H2')
plt.savefig('4c4.svg', bbox_inches='tight', dpi=600)
# plt.show()

svg1 = "4c1.svg"
svg2 = "4c2.svg"
svg3 = "4c3.svg"
svg4 = "4c4.svg"

w1, h1 = get_svg_size(svg1)
w2, h2 = get_svg_size(svg2)
w3, h3 = get_svg_size(svg3)
w4, h4 = get_svg_size(svg4)

total_width = max( (w1 + w2), (w3 + w4) )
total_height = h1 + h2
Figure(f"{total_width}px", f"{total_height}px",
    SVG(svg1).move(0, 0),
    SVG(svg2).move(total_width - w2, 0),  # shift right
    SVG(svg3).move(0, h1),  # shift down by the height of the first row
    SVG(svg4).move(total_width - w4, h1)
).save("figS1.svg")

cairosvg.svg2pdf(url='figS1.svg', write_to='figS1.pdf')